In [ ]:
!pip install -q -U google-generativeai

In [ ]:
# prompt: mount my google drive
import os
from google.colab import userdata
from google.colab import drive
import google.generativeai as genai
from IPython.display import Markdown
import os
import time


In [ ]:
drive.mount('/content/drive', force_remount=True)
#os.chdir('/content/drive/My Drive/Advanced AI/Fowl/')
os.chdir('/content/drive/My Drive/ACERVULINA(ACE)')

Mounted at /content/drive


In [ ]:
GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')

genai.configure(api_key=GOOGLE_API_KEY)

In [ ]:
model = genai.GenerativeModel(model_name="gemini-1.5-flash")

In [ ]:
sample_file = genai.upload_file(path="/content/drive/My Drive/ACERVULINA(ACE)/ACE5291.jpg",
                            display_name="ACE5291 test")

In [ ]:
print(f"Uploaded file: {sample_file.display_name}")

In [ ]:
response = model.generate_content([sample_file, "This image belongs to a parasite, can you identify which class of parasite it belongs to? Answer in one word only."])
Markdown(">" + response.text)

>Trematoda


In [ ]:
def process_images_in_folder(folder_path):
    results = []

    # List all files directly in the given folder (no subfolders)
    files = os.listdir(folder_path)

    for image_file in files:
        # Check if the file is an image (optional filtering)
        if len(results) > 650:
          break
        if not image_file.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue

        image_path = os.path.join(folder_path, image_file)

        # Retry mechanism with exponential backoff
        max_retries = 6
        retries = 0
        success = False
        while retries < max_retries and not success:
            try:
                # Uploading the file to some API
                sample_file = genai.upload_file(path=image_path, display_name=image_file)
                print(f"Uploaded {image_path}")

                # Asking the model for a classification
                response = model.generate_content([
                    sample_file,
                "This image belongs to a parasite, can you identify which class of parasite it belongs to? Answer in one word only"
                ])

                # Storing the result
                results.append({
                    "image": image_path,
                    "classification": response.text.strip()
                })
                print(f"the response for {image_file} is : {response.text.strip()} the total number of results is {len(results)}")
                success = True
            except Exception as e:
                print(f"Error processing {image_path}: {e}. Retrying in {2**retries} seconds...")
                time.sleep(2**retries)
                retries += 1

        # If all retries failed
        if not success:
            print(f"Failed to process {image_path} after multiple retries.")
            results.append({
                "image": image_path,
                "classification": "Error"
            })

    return results

results = process_images_in_folder('/content/drive/My Drive/NECATRİX(NEC)')
results

In [ ]:
import csv
with open('my_list.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    for item in results:
        writer.writerow([item])

from google.colab import files
files.download('my_list.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>